# TCC-04 — Sprint 2 (continuação): v6, v5 com fotos clínicas sobreamostradas + limiar por modalidade

**Projeto:** Tonare — classificador de melanoma (mel vs nv) com fairness por tom de pele
**Framework:** PyTorch | **Treino real:** Kaggle Notebooks (GPU) | **Local (CPU):** só smoke test

## O que muda nesta versão (só duas coisas em relação à v5)

1. **Fotos clínicas do MILK10k repetidas 4x no treino** (`scripts/build_v6_manifest.py`): ~4% do treino
   na v5 -> ~16% na v6. Hipótese: a v4 (clínicas ~10%) foi a melhor em foto clínica (AUC 0,79) e a v5
   (BCN20000 diluiu para ~4%) foi a melhor em dermatoscopia real (MSK 2020, AUC 0,76); sobreamostrar
   as clínicas pode manter os dois ganhos. **Não está provado** — é isso que a v6 testa.
2. **Limiares por modalidade**, escolhidos só na validação (célula 8) e salvos no checkpoint em
   `thresholds_by_modality`. O limiar único em `threshold` continua salvo, como antes.

Splits de val/teste/`test_milk` idênticos aos da v4/v5 (a comparação v4 x v5 x v6 é direta).
O augmentation continua aleatório a cada época, então as cópias repetidas não são vistas iguais.

### Como rodar no Kaggle
Basta ADICIONAR `split_index_v6.csv` ao dataset `tonare-tcc1` (as imagens são as mesmas da v5).
Ligue GPU + Internet e use Run All.


In [1]:
import os
import sys
from pathlib import Path

# ---- conteúdo de scripts/torch_dataset.py (embutido: import relativo não funciona no Kaggle) ----
"""
Dataset PyTorch reutilizável para o classificador mel vs nv (HAM10000).

Encapsula duas decisões importantes documentadas na Sprint 1
(ver scripts/preprocess.py e scripts/split_dataset.py):

1. Imagens aumentadas (is_augmented=True) só devem ser usadas em treino. Usar augmentation
   em val/teste mediria a performance do modelo sobre dado sintético, não sobre a distribuição
   real que ele vai encontrar em produção — por isso `build_datasets()` filtra isso
   automaticamente por split.
2. O split já foi feito por LESÃO (não por imagem) — este módulo não precisa se preocupar com
   isso de novo, só consome a coluna `split` do índice.

Uso:
    from torch_dataset import build_datasets, build_dataloaders

    datasets = build_datasets(base_dir=Path("."))
    loaders = build_dataloaders(datasets, batch_size=32)
"""

from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
LABEL_MAP = {"nv": 0, "mel": 1}
CLASS_NAMES = ["nv", "mel"]


def build_transforms(train: bool, strong_aug: bool = False) -> transforms.Compose:
    """Imagens já vêm em 224x224 da Sprint 1 — aqui só normaliza.

    strong_aug=True (treino): augmentation ALEATÓRIO a cada época (rotação livre, flips,
    recorte, brilho/contraste leves). Lesões de pele não têm orientação canônica, então
    rotação/flip preservam o rótulo. Deliberadamente SEM alterar matiz/saturação: a cor é
    sinal diagnóstico do melanoma e é também o atributo que a análise de fairness vai olhar.
    """
    if train and strong_aug:
        return transforms.Compose([
            transforms.RandomRotation(180),
            transforms.RandomResizedCrop(224, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.15, contrast=0.15),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    if train:
        return transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class MelanomaDataset(Dataset):
    def __init__(self, manifest_df: pd.DataFrame, base_dir: Path, transform=None):
        self.df = manifest_df.reset_index(drop=True)
        self.base_dir = Path(base_dir)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        # "path" foi gravado no Windows (separador \) - normaliza pra rodar em Linux/Kaggle também
        img_path = self.base_dir / row["path"].replace("\\", "/")
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = LABEL_MAP[row["dx"]]
        return img, label


def load_split_manifest(base_dir: Path, split_index_path: str | None = None) -> pd.DataFrame:
    base_dir = Path(base_dir)
    path = Path(split_index_path) if split_index_path else base_dir / "data" / "processed" / "split_index.csv"
    df = pd.read_csv(path)
    return df


def build_datasets(base_dir: Path, split_index_path: str | None = None,
                    limit_per_split: int | None = None,
                    use_precomputed_aug: bool = True,
                    strong_aug: bool = False) -> dict[str, MelanomaDataset]:
    """limit_per_split: usar um número pequeno (ex: 20) só para smoke test em CPU local —
    NÃO usar isso pra treino de verdade (isso é só validação de que o pipeline roda sem erro).

    use_precomputed_aug=False: o treino usa só as imagens ORIGINAIS (sem as 5 cópias fixas
    geradas na Sprint 1). Com strong_aug=True cada época vê variações novas, o que evita o
    overfitting por decorar cópias repetidas (visto no treino: recall 0,99 no treino vs 0,5 na val).
    """
    manifest = load_split_manifest(base_dir, split_index_path)

    datasets = {}
    for split in ("train", "val", "test"):
        split_df = manifest[manifest["split"] == split].copy()
        if split != "train" or not use_precomputed_aug:
            # val/test nunca usam imagens aumentadas - ver docstring do módulo
            split_df = split_df[~split_df["is_augmented"]]
        if limit_per_split:
            per_class = max(1, limit_per_split // 2)
            parts = [group.head(per_class) for _, group in split_df.groupby("dx")]
            split_df = pd.concat(parts, ignore_index=True)
        transform = build_transforms(train=(split == "train"), strong_aug=strong_aug)
        datasets[split] = MelanomaDataset(split_df, base_dir, transform=transform)
    return datasets


def build_dataloaders(datasets: dict[str, MelanomaDataset], batch_size: int = 32,
                       num_workers: int = 2) -> dict[str, DataLoader]:
    loaders = {}
    for split, ds in datasets.items():
        loaders[split] = DataLoader(
            ds, batch_size=batch_size, shuffle=(split == "train"),
            num_workers=num_workers, pin_memory=torch.cuda.is_available(),
        )
    return loaders


In [2]:
IS_KAGGLE = os.path.exists("/kaggle/input")

SMOKE_TEST = not IS_KAGGLE      # local (CPU) = smoke test; Kaggle (GPU) = treino real
ARCHITECTURE = "resnet50"       # ou "efficientnet_b0"
SEED = 42

EPOCHS_STAGE1 = 1 if SMOKE_TEST else 5    # máximo; early stopping pode parar antes
EPOCHS_STAGE2 = 1 if SMOKE_TEST else 12   # máximo; early stopping pode parar antes
PATIENCE = 3                              # épocas sem melhora de AUC antes de parar
BATCH_SIZE = 4 if SMOKE_TEST else 32
LIMIT_PER_SPLIT = 40 if SMOKE_TEST else None
NUM_WORKERS = 0 if SMOKE_TEST else 4

MEL_EXTRA_MULTIPLIER = 1.0   # peso extra pra mel ALÉM do balanceamento (mexer só depois de ver resultados)
TARGET_RECALL_MEL = 0.90     # meta clínica de sensibilidade, usada só pra escolher o limiar na VALIDAÇÃO

print(f"IS_KAGGLE={IS_KAGGLE} | SMOKE_TEST={SMOKE_TEST} | arquitetura={ARCHITECTURE} | batch_size={BATCH_SIZE}")


IS_KAGGLE=True | SMOKE_TEST=False | arquitetura=resnet50 | batch_size=32


In [3]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from torch import optim
from torchvision import models

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


Device: cuda


## 1. Dados e dataloader

No Kaggle o dataset chega sem a pasta `data/` por cima (`.../processed/...`), então criamos um
link simbólico em `/kaggle/working/data/processed` para o código achar `data/processed/...`.

In [7]:
def find_processed():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "split_index_v5.csv" in files:
            return root
        dirs[:] = [d for d in dirs if d != "images_224"]   # não desce nas 31 mil imagens
    return None

if IS_KAGGLE:
    KAGGLE_PROCESSED = find_processed()
    assert KAGGLE_PROCESSED, (f"split_index_v5.csv não encontrado. Conteúdo de /kaggle/input: "
                              f"{os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'pasta não existe'}"
                              " - anexe o dataset tonare-tcc1 (Add Input).")
    print("Dataset encontrado em:", KAGGLE_PROCESSED)
    os.makedirs("/kaggle/working/data", exist_ok=True)
    link = "/kaggle/working/data/processed"
    if os.path.islink(link) or os.path.exists(link):
        os.remove(link)
    os.symlink(KAGGLE_PROCESSED, link)
    BASE_DIR = Path("/kaggle/working")
    OUT_DIR = Path("/kaggle/working")
else:
    BASE_DIR = Path("..")
    OUT_DIR = BASE_DIR / "data" / "processed"

# v6 = v5 + fotos clínicas do MILK10k repetidas 4x no treino (mesma regra de scripts/build_v6_manifest.py)
_v5 = pd.read_csv(BASE_DIR / "data" / "processed" / "split_index_v5.csv")
_clin = (_v5["dataset"] == "milk10k") & (_v5["image_type"] == "clinical") & (_v5["split"] == "train")
_v6 = pd.concat([_v5] + [_v5[_clin]] * 3, ignore_index=True)
SPLIT_INDEX_PATH = OUT_DIR / "split_index_v6.csv"
_v6.to_csv(SPLIT_INDEX_PATH, index=False)

print("BASE_DIR:", BASE_DIR, "| OUT_DIR:", OUT_DIR, "| manifesto:", SPLIT_INDEX_PATH.name)
print("treino:", int((_v6["split"] == "train").sum()), "imagens (esperado: 18075)")

Dataset encontrado em: /kaggle/input/datasets/sofyaoliveiras/tonare-tcc-v2/processed (2)/processed
BASE_DIR: /kaggle/working | OUT_DIR: /kaggle/working | manifesto: split_index_v6.csv
treino: 18075 imagens (esperado: 18075)


In [8]:
datasets = build_datasets(base_dir=BASE_DIR, split_index_path=SPLIT_INDEX_PATH,
                          limit_per_split=LIMIT_PER_SPLIT, use_precomputed_aug=False, strong_aug=True)
loaders = build_dataloaders(datasets, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

for split, ds in datasets.items():
    contagem = ds.df["dx"].value_counts().to_dict()
    print(f"{split}: {len(ds)} imagens {contagem}")


train: 18075 imagens {'nv': 11816, 'mel': 6259}
val: 2458 imagens {'nv': 1730, 'mel': 728}
test: 1166 imagens {'nv': 1001, 'mel': 165}


In [9]:
imgs, labels = next(iter(loaders["train"]))
print("Shape do batch de imagens:", imgs.shape)
print("Labels do batch:", labels.tolist())
print("Range de valores:", float(imgs.min()), "a", float(imgs.max()))

assert imgs.shape[1:] == (3, 224, 224), "dataloader não está entregando imagens 224x224x3"
print("\nDataloader validado: uma iteração de teste passou sem erro.")


Shape do batch de imagens: torch.Size([32, 3, 224, 224])
Labels do batch: [1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0]
Range de valores: -2.1179039478302 a 2.640000104904175

Dataloader validado: uma iteração de teste passou sem erro.


## 2. Modelo pré-treinado

In [10]:
def build_model(architecture: str, num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    if architecture == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif architecture == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Arquitetura não suportada: {architecture}")
    return model


def set_backbone_trainable(model: nn.Module, architecture: str, trainable: bool) -> None:
    # Congela/descongela tudo, MENOS a camada final de classificação.
    for param in model.parameters():
        param.requires_grad = trainable
    head = model.fc if architecture == "resnet50" else model.classifier
    for param in head.parameters():
        param.requires_grad = True


model = build_model(ARCHITECTURE).to(DEVICE)
set_backbone_trainable(model, ARCHITECTURE, trainable=False)

n_treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis (Estágio 1): {n_treinaveis:,} de {n_total:,} total")


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 215MB/s]


Parâmetros treináveis (Estágio 1): 4,098 de 23,512,130 total


## 3. Loss ponderada

O treino agora tem só imagens originais (~1:6 mel:nv), então o peso "balanced" já dá ~6x mais
importância ao erro em mel. `MEL_EXTRA_MULTIPLIER` (default 1.0) permite reforçar ainda mais.

In [11]:
train_labels = datasets["train"].df["dx"].map(LABEL_MAP).values
base_weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=train_labels)
class_weights = base_weights.copy()
class_weights[1] *= MEL_EXTRA_MULTIPLIER
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32).to(DEVICE))

print(f"Distribuição do treino: {pd.Series(train_labels).value_counts().to_dict()} (0=nv, 1=mel)")
print(f"Pesos da loss: nv={class_weights[0]:.3f}, mel={class_weights[1]:.3f}")
if not SMOKE_TEST:  # o smoke test é balanceado por construção (20 mel/20 nv)
    assert class_weights[1] > class_weights[0], "peso de mel deveria ser maior que o de nv"


Distribuição do treino: {0: 11816, 1: 6259} (0=nv, 1=mel)
Pesos da loss: nv=0.765, mel=1.444


## 4. Loop de treino (early stopping pela AUC de validação)

In [12]:
def run_epoch(model, loader, criterion, optimizer=None) -> dict:
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            all_probs.extend(F.softmax(outputs.detach(), dim=1)[:, 1].cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    labels_arr, probs_arr = np.array(all_labels), np.array(all_probs)
    preds = (probs_arr >= 0.5).astype(int)
    report = classification_report(labels_arr, preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    auc = roc_auc_score(labels_arr, probs_arr) if len(set(all_labels)) > 1 else float("nan")
    return {"loss": total_loss / len(loader.dataset), "accuracy": report["accuracy"],
            "recall_mel": report["mel"]["recall"], "auc": auc,
            "labels": labels_arr, "probs": probs_arr}


def train_stage(model, loaders, criterion, optimizer, n_epochs: int, stage_name: str, patience: int = PATIENCE):
    history, best_auc, best_state, sem_melhora = [], -1.0, None, 0
    print(f"[{stage_name}] pesos da loss (nv, mel): {[round(w, 3) for w in criterion.weight.tolist()]}")
    for epoch in range(1, n_epochs + 1):
        tr = run_epoch(model, loaders["train"], criterion, optimizer)
        va = run_epoch(model, loaders["val"], criterion, optimizer=None)
        print(f"[{stage_name}] época {epoch}/{n_epochs} | "
              f"treino: loss={tr['loss']:.3f} auc={tr['auc']:.3f} | "
              f"val: loss={va['loss']:.3f} auc={va['auc']:.3f} recall_mel@0.5={va['recall_mel']:.3f}")
        history.append({"epoch": epoch, "train": tr, "val": va})

        if va["auc"] > best_auc:
            best_auc, best_state, sem_melhora = va["auc"], copy.deepcopy(model.state_dict()), 0
        else:
            sem_melhora += 1
            if sem_melhora >= patience:
                print(f"[{stage_name}] early stopping: sem melhora da AUC de validação há {patience} épocas")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"[{stage_name}] modelo restaurado para a melhor época (val AUC = {best_auc:.3f})")
    return history


## 5. Estágio 1 — só a camada final
Backbone congelado, LR alto (1e-3) porque são poucos parâmetros novos.

In [13]:
optimizer_stage1 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-2)
history_stage1 = train_stage(model, loaders, criterion, optimizer_stage1, EPOCHS_STAGE1, "Estágio 1")


[Estágio 1] pesos da loss (nv, mel): [0.765, 1.444]
[Estágio 1] época 1/5 | treino: loss=0.507 auc=0.830 | val: loss=0.443 auc=0.854 recall_mel@0.5=0.728
[Estágio 1] época 2/5 | treino: loss=0.473 auc=0.854 | val: loss=0.443 auc=0.856 recall_mel@0.5=0.738
[Estágio 1] época 3/5 | treino: loss=0.471 auc=0.855 | val: loss=0.443 auc=0.857 recall_mel@0.5=0.735
[Estágio 1] época 4/5 | treino: loss=0.460 auc=0.863 | val: loss=0.442 auc=0.856 recall_mel@0.5=0.652
[Estágio 1] época 5/5 | treino: loss=0.465 auc=0.860 | val: loss=0.462 auc=0.853 recall_mel@0.5=0.765
[Estágio 1] modelo restaurado para a melhor época (val AUC = 0.857)


## 6. Estágio 2 — fine-tuning profundo
Backbone descongelado, LR bem baixo (1e-5) + weight decay para regularizar.

In [14]:
set_backbone_trainable(model, ARCHITECTURE, trainable=True)
n_treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros treináveis (Estágio 2): {n_treinaveis:,}")

optimizer_stage2 = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)
history_stage2 = train_stage(model, loaders, criterion, optimizer_stage2, EPOCHS_STAGE2, "Estágio 2")


Parâmetros treináveis (Estágio 2): 23,512,130
[Estágio 2] pesos da loss (nv, mel): [0.765, 1.444]
[Estágio 2] época 1/12 | treino: loss=0.447 auc=0.871 | val: loss=0.424 auc=0.869 recall_mel@0.5=0.739
[Estágio 2] época 2/12 | treino: loss=0.420 auc=0.886 | val: loss=0.405 auc=0.880 recall_mel@0.5=0.738
[Estágio 2] época 3/12 | treino: loss=0.395 auc=0.901 | val: loss=0.391 auc=0.885 recall_mel@0.5=0.712
[Estágio 2] época 4/12 | treino: loss=0.377 auc=0.909 | val: loss=0.384 auc=0.891 recall_mel@0.5=0.698
[Estágio 2] época 5/12 | treino: loss=0.360 auc=0.918 | val: loss=0.383 auc=0.894 recall_mel@0.5=0.734
[Estágio 2] época 6/12 | treino: loss=0.335 auc=0.930 | val: loss=0.386 auc=0.894 recall_mel@0.5=0.731
[Estágio 2] época 7/12 | treino: loss=0.325 auc=0.934 | val: loss=0.384 auc=0.897 recall_mel@0.5=0.753
[Estágio 2] época 8/12 | treino: loss=0.308 auc=0.941 | val: loss=0.388 auc=0.896 recall_mel@0.5=0.736
[Estágio 2] época 9/12 | treino: loss=0.289 auc=0.949 | val: loss=0.391 auc=0.

## 7. Escolha do limiar (só na validação)

O modelo dá uma probabilidade de melanoma; o limiar decide a partir de quanto marcamos como `mel`.
Baixar o limiar aumenta a sensibilidade (menos melanomas perdidos) e custa especificidade
(mais nevos benignos marcados). A tabela mostra esse custo.

In [15]:
def sens_spec_prec(labels, probs, threshold):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    prec = tp / (tp + fp) if (tp + fp) else float("nan")
    return sens, spec, prec


val_res = run_epoch(model, loaders["val"], criterion)
val_labels, val_probs = val_res["labels"], val_res["probs"]
print(f"Validação: AUC-ROC = {val_res['auc']:.3f}\n")

print(f"{'limiar':>7} {'sensibilidade(mel)':>19} {'especificidade':>15} {'precisão(mel)':>14}")
for t in np.arange(0.05, 0.96, 0.05):
    sens, spec, prec = sens_spec_prec(val_labels, val_probs, t)
    print(f"{t:>7.2f} {sens:>19.3f} {spec:>15.3f} {prec:>14.3f}")


Validação: AUC-ROC = 0.901

 limiar  sensibilidade(mel)  especificidade  precisão(mel)
   0.05               0.952           0.556          0.474
   0.10               0.916           0.651          0.525
   0.15               0.886           0.703          0.557
   0.20               0.872           0.745          0.590
   0.25               0.859           0.776          0.618
   0.30               0.841           0.808          0.648
   0.35               0.821           0.827          0.666
   0.40               0.798           0.846          0.685
   0.45               0.771           0.872          0.717
   0.50               0.745           0.883          0.728
   0.55               0.706           0.899          0.746
   0.60               0.684           0.914          0.771
   0.65               0.650           0.924          0.782
   0.70               0.615           0.933          0.794
   0.75               0.582           0.943          0.812
   0.80               0.541 

## 8. Resultado final no teste (olhado uma única vez)

Limiar escolhido na **validação**: o maior limiar que ainda atinge `TARGET_RECALL_MEL`
(maior limiar = menos falsos positivos dentro da meta). Se nenhum atinge, usa o de maior
sensibilidade e avisa.

In [16]:
thresholds = np.round(np.arange(0.01, 0.99, 0.01), 2)
sens_por_t = {t: sens_spec_prec(val_labels, val_probs, t)[0] for t in thresholds}
candidatos = [t for t, s in sens_por_t.items() if s >= TARGET_RECALL_MEL]

if candidatos:
    best_threshold = max(candidatos)
    print(f"Limiar escolhido na validação: {best_threshold:.2f} (sensibilidade >= {TARGET_RECALL_MEL})")
else:
    best_threshold = max(sens_por_t, key=lambda t: (sens_por_t[t], t))  # maior sensibilidade; empate -> maior limiar
    print(f"AVISO: nenhum limiar atinge sensibilidade {TARGET_RECALL_MEL} na validação. "
          f"Usando o de maior sensibilidade: {best_threshold:.2f} ({sens_por_t[best_threshold]:.3f})")

test_res = run_epoch(model, loaders["test"], criterion)
test_labels, test_probs = test_res["labels"], test_res["probs"]
print(f"\nTESTE — AUC-ROC = {test_res['auc']:.3f}")
for nome, t in [("limiar padrão 0.50", 0.5), (f"limiar escolhido {best_threshold:.2f}", best_threshold)]:
    sens, spec, prec = sens_spec_prec(test_labels, test_probs, t)
    tn, fp, fn, tp = confusion_matrix(test_labels, (test_probs >= t).astype(int), labels=[0, 1]).ravel()
    print(f"  {nome}: sensibilidade(mel)={sens:.3f} especificidade={spec:.3f} precisão(mel)={prec:.3f} "
          f"| melanomas perdidos: {fn} de {tp + fn} | nevos marcados como mel: {fp} de {tn + fp}")


# --- Limiares POR MODALIDADE (escolhidos só na validação; o teste não entra aqui) ---
# loaders["val"] não embaralha, então a ordem das probabilidades é a ordem de datasets["val"].df
val_mod = datasets["val"].df["image_type"].values
assert len(val_mod) == len(val_labels), "ordem da validação não bate com o dataframe"

thr_mod = {}
for mod in ("dermoscopic", "clinical"):
    m = val_mod == mod
    lab, prb = val_labels[m], val_probs[m]
    sens_t = {t: sens_spec_prec(lab, prb, t)[0] for t in thresholds}
    ok = [t for t, s in sens_t.items() if s >= TARGET_RECALL_MEL]
    thr_a = max(ok) if ok else max(sens_t, key=lambda t: (sens_t[t], t))      # sensibilidade >= meta
    youden = {t: sum(sens_spec_prec(lab, prb, t)[:2]) - 1 for t in thresholds}
    thr_b = max(youden, key=lambda t: (youden[t], t))                          # equilibrado (Youden)
    thr_mod[mod] = {"A": float(thr_a), "B": float(thr_b)}
    print(f"Validação {mod}: {int(lab.sum())} mel / {int((lab == 0).sum())} nv -> "
          f"limiar A (sens>={TARGET_RECALL_MEL})={thr_a:.2f} | limiar B (Youden)={thr_b:.2f}")

print()
print("AVISO: a validação clínica tem poucos casos; trate o limiar clínico como ruidoso.")
print("Teste do HAM10000 (só dermatoscopia) com os limiares de dermatoscopia:")
for nome in ("A", "B"):
    t = thr_mod["dermoscopic"][nome]
    sens, spec, prec = sens_spec_prec(test_labels, test_probs, t)
    print(f"  {nome} (limiar {t:.2f}): sensibilidade={sens:.3f} especificidade={spec:.3f} precisão={prec:.3f}")


Limiar escolhido na validação: 0.14 (sensibilidade >= 0.9)

TESTE — AUC-ROC = 0.908
  limiar padrão 0.50: sensibilidade(mel)=0.582 especificidade=0.954 precisão(mel)=0.676 | melanomas perdidos: 69 de 165 | nevos marcados como mel: 46 de 1001
  limiar escolhido 0.14: sensibilidade(mel)=0.782 especificidade=0.823 precisão(mel)=0.422 | melanomas perdidos: 36 de 165 | nevos marcados como mel: 177 de 1001
Validação dermoscopic: 684 mel / 1656 nv -> limiar A (sens>=0.9)=0.14 | limiar B (Youden)=0.31
Validação clinical: 44 mel / 74 nv -> limiar A (sens>=0.9)=0.01 | limiar B (Youden)=0.31

AVISO: a validação clínica tem poucos casos; trate o limiar clínico como ruidoso.
Teste do HAM10000 (só dermatoscopia) com os limiares de dermatoscopia:
  A (limiar 0.14): sensibilidade=0.782 especificidade=0.823 precisão=0.422
  B (limiar 0.31): sensibilidade=0.667 especificidade=0.910 precisão=0.550


In [17]:
checkpoint_path = OUT_DIR / f"model_{ARCHITECTURE}_baseline_v6.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "architecture": ARCHITECTURE,
    "label_map": LABEL_MAP,
    "threshold": float(best_threshold),
    "target_recall_mel": TARGET_RECALL_MEL,
    "mel_extra_multiplier": MEL_EXTRA_MULTIPLIER,
    "thresholds_by_modality": thr_mod,
    "val_auc": float(val_res["auc"]),
    "smoke_test": SMOKE_TEST,
}, checkpoint_path)
print(f"Checkpoint salvo em: {checkpoint_path}")
if SMOKE_TEST:
    print("SMOKE TEST (CPU, amostra pequena): serve só para provar que o pipeline roda. "
          "Rode no Kaggle (GPU) para o treino de verdade.")


Checkpoint salvo em: /kaggle/working/model_resnet50_baseline_v6.pt


## Próximo passo
Baixar `model_resnet50_baseline_v6.pt` para `data/processed/` e comparar com v4/v5 nos mesmos testes:

```
python scripts/evaluate_milk10k.py v6 heldout
python scripts/evaluate_padufes20.py v6
python scripts/evaluate_external.py v6 hiba251
python scripts/evaluate_external.py v6 consec_mel2020_216
python scripts/evaluate_ddi.py v6
python scripts/evaluate_ham_test.py v6
python scripts/threshold_by_modality.py v4 v5 v6
```
